# Underspecification Impact Analysis

Comparing pass@k and pass^k metrics between:
- **Original Baseline** (`underspec_baseline_og`)
- **Severe Underspecification** (`underspec_baseline_severe`)

Model: `gemini-3-pro-preview`

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Load the data
og_path = 'runs/run_underspec_baseline_og_gemini-3-pro-preview_20260117_001728/pass_k_per_task.csv'
severe_path = 'runs/run_underspec_baseline_severe_gemini-3-pro-preview_20260117_002322/pass_k_per_task.csv'

df_og = pd.read_csv(og_path)
df_severe = pd.read_csv(severe_path)

print(f"Original baseline: {len(df_og) - 1} tasks")
print(f"Severe underspec: {len(df_severe) - 1} tasks")

## Aggregate Metrics Comparison

In [ ]:
# Extract aggregate rows
agg_og = df_og[df_og['task_id'] == 'AGGREGATE'].iloc[0]
agg_severe = df_severe[df_severe['task_id'] == 'AGGREGATE'].iloc[0]

# Create comparison dataframe
metrics = ['pass_rate', 'pass^1', 'pass^2', 'pass^3', 'pass@1', 'pass@2', 'pass@3']
comparison = pd.DataFrame({
    'Metric': metrics,
    'Original': [agg_og[m] for m in metrics],
    'Severe Underspec': [agg_severe[m] for m in metrics]
})
comparison['Delta'] = comparison['Severe Underspec'] - comparison['Original']
comparison['% Change'] = (comparison['Delta'] / comparison['Original'] * 100).round(1)

print("\n" + "="*70)
print("AGGREGATE METRICS COMPARISON")
print("="*70)
comparison

In [ ]:
# Visualize aggregate comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart comparison
x = np.arange(len(metrics))
width = 0.35

ax1 = axes[0]
bars1 = ax1.bar(x - width/2, comparison['Original'], width, label='Original', color='#2ecc71', alpha=0.8)
bars2 = ax1.bar(x + width/2, comparison['Severe Underspec'], width, label='Severe Underspec', color='#e74c3c', alpha=0.8)

ax1.set_ylabel('Score')
ax1.set_title('pass@k and pass^k Metrics Comparison')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics, rotation=45, ha='right')
ax1.legend()
ax1.set_ylim(0, 1.1)
ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax1.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    height = bar.get_height()
    ax1.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)

# Delta chart
ax2 = axes[1]
colors = ['#e74c3c' if d < 0 else '#2ecc71' for d in comparison['Delta']]
bars3 = ax2.bar(metrics, comparison['Delta'], color=colors, alpha=0.8)
ax2.set_ylabel('Change in Score')
ax2.set_title('Performance Drop (Severe - Original)')
ax2.set_xticklabels(metrics, rotation=45, ha='right')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Add value labels
for bar, delta in zip(bars3, comparison['Delta']):
    height = bar.get_height()
    ax2.annotate(f'{delta:+.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, -12 if height < 0 else 3), textcoords='offset points', 
                ha='center', va='top' if height < 0 else 'bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Per-Task Analysis

In [ ]:
# Merge per-task data
df_og_tasks = df_og[df_og['task_id'] != 'AGGREGATE'].copy()
df_severe_tasks = df_severe[df_severe['task_id'] != 'AGGREGATE'].copy()

merged = df_og_tasks.merge(df_severe_tasks, on='task_id', suffixes=('_og', '_severe'))
merged['pass_rate_delta'] = merged['pass_rate_severe'] - merged['pass_rate_og']
merged['pass@1_delta'] = merged['pass@1_severe'] - merged['pass@1_og']
merged['pass^1_delta'] = merged['pass^1_severe'] - merged['pass^1_og']

print(f"Tasks with decreased performance: {(merged['pass_rate_delta'] < 0).sum()} / {len(merged)}")
print(f"Tasks with unchanged performance: {(merged['pass_rate_delta'] == 0).sum()} / {len(merged)}")
print(f"Tasks with improved performance: {(merged['pass_rate_delta'] > 0).sum()} / {len(merged)}")

In [ ]:
# Detailed per-task comparison
task_comparison = merged[['task_id', 'pass_rate_og', 'pass_rate_severe', 'pass_rate_delta', 
                          'pass@1_og', 'pass@1_severe', 'pass^1_og', 'pass^1_severe']].copy()
task_comparison = task_comparison.sort_values('pass_rate_delta')

print("\n" + "="*70)
print("PER-TASK COMPARISON (sorted by performance delta)")
print("="*70)
task_comparison

In [ ]:
# Visualize per-task changes
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter plot: Original vs Severe pass rates
ax1 = axes[0]
ax1.scatter(merged['pass_rate_og'], merged['pass_rate_severe'], alpha=0.7, s=100, c='#3498db', edgecolors='white')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='No change line')
ax1.set_xlabel('Original Pass Rate')
ax1.set_ylabel('Severe Underspec Pass Rate')
ax1.set_title('Per-Task Pass Rate Comparison')
ax1.set_xlim(-0.05, 1.05)
ax1.set_ylim(-0.05, 1.05)
ax1.legend()

# Annotate points below the diagonal
for _, row in merged.iterrows():
    if row['pass_rate_severe'] < row['pass_rate_og'] - 0.1:
        ax1.annotate(row['task_id'][:8] + '...', 
                    (row['pass_rate_og'], row['pass_rate_severe']),
                    fontsize=7, alpha=0.7)

# Histogram of deltas
ax2 = axes[1]
ax2.hist(merged['pass_rate_delta'], bins=10, color='#9b59b6', alpha=0.7, edgecolor='white')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax2.axvline(x=merged['pass_rate_delta'].mean(), color='red', linestyle='-', linewidth=2, 
           label=f'Mean: {merged["pass_rate_delta"].mean():.2f}')
ax2.set_xlabel('Pass Rate Change')
ax2.set_ylabel('Number of Tasks')
ax2.set_title('Distribution of Pass Rate Changes')
ax2.legend()

plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)

print(f"\nOverall Performance Impact:")
print(f"   - Original pass rate: {agg_og['pass_rate']:.1%}")
print(f"   - Severe underspec pass rate: {agg_severe['pass_rate']:.1%}")
print(f"   - Absolute drop: {agg_severe['pass_rate'] - agg_og['pass_rate']:+.1%}")
print(f"   - Relative drop: {((agg_severe['pass_rate'] - agg_og['pass_rate']) / agg_og['pass_rate'] * 100):+.1f}%")

print(f"\npass@1 (probability of at least one success):")
print(f"   - Original: {agg_og['pass@1']:.1%}")
print(f"   - Severe: {agg_severe['pass@1']:.1%}")
print(f"   - Change: {agg_severe['pass@1'] - agg_og['pass@1']:+.1%}")

print(f"\npass^1 (probability all attempts succeed):")
print(f"   - Original: {agg_og['pass^1']:.1%}")
print(f"   - Severe: {agg_severe['pass^1']:.1%}")
print(f"   - Change: {agg_severe['pass^1'] - agg_og['pass^1']:+.1%}")

print(f"\nTask-level impact:")
print(f"   - Tasks degraded: {(merged['pass_rate_delta'] < 0).sum()} ({(merged['pass_rate_delta'] < 0).sum()/len(merged)*100:.1f}%)")
print(f"   - Tasks unchanged: {(merged['pass_rate_delta'] == 0).sum()} ({(merged['pass_rate_delta'] == 0).sum()/len(merged)*100:.1f}%)")
print(f"   - Tasks improved: {(merged['pass_rate_delta'] > 0).sum()} ({(merged['pass_rate_delta'] > 0).sum()/len(merged)*100:.1f}%)")
print(f"   - Mean pass rate delta: {merged['pass_rate_delta'].mean():.2%}")

In [ ]:
# Identify most affected tasks
print("\n" + "="*70)
print("MOST AFFECTED TASKS (largest performance drops)")
print("="*70)

most_affected = merged.nsmallest(5, 'pass_rate_delta')[['task_id', 'pass_rate_og', 'pass_rate_severe', 'pass_rate_delta']]
for _, row in most_affected.iterrows():
    print(f"\n  {row['task_id']}:")
    print(f"    Original: {row['pass_rate_og']:.1%} -> Severe: {row['pass_rate_severe']:.1%} (Delta = {row['pass_rate_delta']:+.1%})")

In [ ]:
# Robust tasks (maintained performance)
print("\n" + "="*70)
print("ROBUST TASKS (maintained or improved performance)")
print("="*70)

robust = merged[merged['pass_rate_delta'] >= 0][['task_id', 'pass_rate_og', 'pass_rate_severe', 'pass_rate_delta']]
for _, row in robust.iterrows():
    print(f"\n  {row['task_id']}:")
    print(f"    Original: {row['pass_rate_og']:.1%} -> Severe: {row['pass_rate_severe']:.1%} (Delta = {row['pass_rate_delta']:+.1%})")